In [14]:
"""
Baseline Modeling Pipeline — WV Opioid Data
Target: CHANGE in LA_Tot_Opioid_Clms (next year minus current year)

Features: raw socioeconomic + YoY deltas for all features + poverty x rucc
Models:   Ridge, ElasticNet, Random Forest, Gradient Boosting, XGBoost
"""

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils import resample
from xgboost import XGBRegressor

In [15]:
# ── 1. Load & sort ─────────────────────────────────────────────
df = pd.read_csv("../data/west_virginia_opioid_data_clean.csv")
df = df.sort_values(['FIPS_Code', 'Year']).reset_index(drop=True)

In [16]:
# ── 2. Target (LEVEL, not change) ──────────────────────────────
df['target'] = df.groupby('FIPS_Code')['LA_Tot_Opioid_Clms'].shift(-1)

In [17]:
# ── 3. Lag features (CRITICAL) ─────────────────────────────────
df['claims_lag1'] = df.groupby('FIPS_Code')['LA_Tot_Opioid_Clms'].shift(1)
df['claims_lag2'] = df.groupby('FIPS_Code')['LA_Tot_Opioid_Clms'].shift(2)
# Rolling stats (only using past data → no leakage)
df['rolling_mean_3'] = (
    df.groupby('FIPS_Code')['LA_Tot_Opioid_Clms']
    .shift(1).rolling(3).mean()
)
df['rolling_std_3'] = (
    df.groupby('FIPS_Code')['LA_Tot_Opioid_Clms']
    .shift(1).rolling(3).std()
)
# Trend (approx slope)
df['trend_3'] = (
    df.groupby('FIPS_Code')['LA_Tot_Opioid_Clms']
    .shift(1).diff().rolling(3).mean()
)

In [18]:
# ── 4. Keep your socioecon features ────────────────────────────
base_features = [
    'Labor_Force_Participation_Rate',
    'Unemployment_Rate',
    'Median_Household_Income',
    'Poverty_Percent_All_Ages',
    'Pct_Bachelors_Plus',
    'Rural_Urban_Continuum_Code'
]
time_features = [
    'claims_lag1',
    'claims_lag2',
    'rolling_mean_3',
    'rolling_std_3',
    'trend_3'
]
feature_cols = base_features + time_features

In [19]:
# ── 5. Drop NA rows safely ─────────────────────────────────────
df_model = df.dropna(subset=feature_cols + ['target']).reset_index(drop=True)

In [20]:
# ── 6. Model (focus on XGBoost) ────────────────────────────────
model = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42
)

In [21]:
# ── 7. LOYO CV (clean) ─────────────────────────────────────────
years = sorted(df_model['Year'].unique())
results = []

for test_year in years[-3:]:
    train = df_model[df_model['Year'] < test_year]
    test  = df_model[df_model['Year'] == test_year]

    X_train, y_train = train[feature_cols], train['target']
    X_test, y_test   = test[feature_cols], test['target']

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    results.append({
        'year': test_year,
        'r2': r2_score(y_test, preds),
        'rmse': root_mean_squared_error(y_test, preds),
        'mae': mean_absolute_error(y_test, preds)
    })

print(pd.DataFrame(results))

   year        r2        rmse         mae
0  2020  0.905754  359.933802  135.894380
1  2021  0.818167  458.623058  163.381192
2  2022  0.843329  303.526077  108.879237
